In [1]:
import sys
print(sys.executable)

/Users/dominikpfeifer/Projects/german-law-obligations/.venv/bin/python


In [2]:
import requests
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)

URL = "https://www.gesetze-im-internet.de/gii-toc.xml"
HEADERS = {"User-Agent": "german-law-obligations (learning project; github.com/pfeifer99)"}

response = requests.get(URL, headers=HEADERS, timeout=30)

print(response.status_code)
print(response.headers.get("Content-Type"))
print(len(response.content), "Bytes")


200
application/xml
1291178 Bytes


In [3]:
(DATA / "gii-toc.xml").write_bytes(response.content)
print(response.text[:1500])

<?xml version="1.0" encoding="UTF-8" ?><!DOCTYPE items SYSTEM "http://www.gesetze-im-internet.de/dtd/1.0/gii-toc.dtd">

<items>

<item>
<title>Gesetz über die Ausprägung einer 1-DM-Goldmünze und die Errichtung der Stiftung &quot;Geld und Währung&quot;</title>
<link>http://www.gesetze-im-internet.de/1-dm-goldm_nzg/xml.zip</link>
</item>
<item>
<title>Erstes Gesetz zur Vereinheitlichung und Neuregelung des Besoldungsrechts in Bund und Ländern</title>
<link>http://www.gesetze-im-internet.de/besvng_1/xml.zip</link>
</item>
<item>
<title>Erste Verordnung zur Durchführung des Bundes-Immissionsschutzgesetzes *)</title>
<link>http://www.gesetze-im-internet.de/bimschv_1_2010/xml.zip</link>
</item>
<item>
<title>Verordnung zur Durchführung von regelmäßigen Datenübermittlungen zwischen Meldebehörden</title>
<link>http://www.gesetze-im-internet.de/bmeldd_v_1_2015/xml.zip</link>
</item>
<item>
<title>Erste Durchführungsverordnung zur Betriebsordnung für Luftfahrtgerät (Anwendungsbestimmungen zu Anh

In [4]:
from lxml import etree
import pandas as pd

tree = etree.parse(str(DATA / "gii-toc.xml"))
root = tree.getroot()
print(root.tag)

items = root.findall("item")
print(len(items), "Einträge")

items
6144 Einträge


In [5]:
rows = []
for item in items:
    rows.append({
        "title": item.findtext("title"),
        "link": item.findtext("link"),
    })

toc = pd.DataFrame(rows)
toc.head()

,title,link
0,Gesetz über die Ausprägung einer 1-DM-Goldmünz...,http://www.gesetze-im-internet.de/1-dm-goldm_n...
1,Erstes Gesetz zur Vereinheitlichung und Neureg...,http://www.gesetze-im-internet.de/besvng_1/xml...
2,Erste Verordnung zur Durchführung des Bundes-I...,http://www.gesetze-im-internet.de/bimschv_1_20...
3,Verordnung zur Durchführung von regelmäßigen D...,http://www.gesetze-im-internet.de/bmeldd_v_1_2...
4,Erste Durchführungsverordnung zur Betriebsordn...,http://www.gesetze-im-internet.de/luftbodv_1_2...


In [6]:
toc["link"] = toc["link"].str.replace("http://", "https://", n=1)
toc["slug"] = toc["link"].str.split("/").str[-2]

print("Zeilen:", len(toc))
print("Eindeutige Kennungen:", toc["slug"].nunique())
print("Fehlende Titel:", toc["title"].isna().sum())

toc.to_csv(DATA / "toc.csv", index=False)

Zeilen: 6144
Eindeutige Kennungen: 6144
Fehlende Titel: 0


In [7]:
import zipfile, io

slug = toc.loc[0, "slug"]
link = toc.loc[0, "link"]
print(slug, link)

r = requests.get(link, headers=HEADERS, timeout=30)
print(r.status_code, len(r.content), "Bytes")

law_dir = DATA / "laws" / slug
law_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    print(z.namelist())
    z.extractall(law_dir)

1-dm-goldm_nzg https://www.gesetze-im-internet.de/1-dm-goldm_nzg/xml.zip
200 3699 Bytes
['BJNR204500000.xml']


In [8]:
xml_file = next(law_dir.glob("*.xml"))
print(xml_file.name)
print(xml_file.read_text(encoding="utf-8")[:3000])

BJNR204500000.xml
<?xml version="1.0" encoding="UTF-8" ?><!DOCTYPE dokumente SYSTEM "http://www.gesetze-im-internet.de/dtd/1.01/gii-norm.dtd">
<dokumente builddate="20260506175647" doknr="BJNR204500000"><norm builddate="20260506175647" doknr="BJNR204500000"><metadaten><jurabk>1-DM-GoldmünzG</jurabk><ausfertigung-datum manuell="ja">2000-12-27</ausfertigung-datum><fundstelle typ="amtlich"><periodikum>BGBl I</periodikum><zitstelle>2000, 2045</zitstelle></fundstelle><langue>Gesetz über die Ausprägung einer 1-DM-Goldmünze
und die Errichtung der Stiftung &quot;Geld und Währung&quot;</langue></metadaten><textdaten><text format="XML"><Content><P/></Content></text><fussnoten><Content><P>(+++ Textnachweis ab: 31.12.2000 +++)</P></Content></fussnoten></textdaten></norm>
<norm builddate="20260506175647" doknr="BJNR204500000BJNG000100305"><metadaten><jurabk>1-DM-GoldmünzG</jurabk><gliederungseinheit><gliederungskennzahl>010</gliederungskennzahl><gliederungsbez>Erster Abschnitt</gliederungsbez><glie

In [9]:
def parse_law(xml_path):
    tree = etree.parse(str(xml_path))
    rows = []
    for norm in tree.getroot().findall("norm"):
        enbez = norm.findtext("metadaten/enbez")
        if enbez is None:
            continue  # Kopf oder Abschnittsüberschrift, kein Paragraf

        content = norm.find("textdaten/text/Content")
        if content is None:
            continue

        text = " ".join(" ".join(content.itertext()).split())
        rows.append({
            "jurabk": norm.findtext("metadaten/jurabk"),
            "enbez": enbez,
            "titel": norm.findtext("metadaten/titel"),
            "text": text,
        })
    return pd.DataFrame(rows)

df = parse_law(xml_file)
df

,jurabk,enbez,titel,text
0,1-DM-GoldmünzG,§ 1,Ausgabe durch die Deutsche Bundesbank,"Die Deutsche Bundesbank wird ermächtigt, zum G..."
1,1-DM-GoldmünzG,§ 2,Gestalt der 1-DM-Goldmünze,(1) Die Gestaltung der Wert- und Bildseite der...
2,1-DM-GoldmünzG,§ 3,Gesetzliches Zahlungsmittel,Die 1-DM-Goldmünzen sind nach Maßgabe des § 4 ...
3,1-DM-GoldmünzG,§ 4,Annahme- und Umtauschpflicht,"(1) Die Deutsche Bundesbank ist verpflichtet, ..."
4,1-DM-GoldmünzG,§ 5,Prägung und Vergütung,(1) Die 1-DM-Goldmünzen werden von denjenigen ...
5,1-DM-GoldmünzG,§ 6,Inverkehrbringen,Die 1-DM-Goldmünzen werden unbeschadet des Art...
6,1-DM-GoldmünzG,§ 7,Ausgabepreis,Die Deutsche Bundesbank bestimmt den Ausgabepr...
7,1-DM-GoldmünzG,§ 8,Erlösverwendung,(1) Der Nettoerlös aus dem Inverkehrbringen de...
8,1-DM-GoldmünzG,§ 9,Anwendung der Bußgeldvorschriften des Münzgese...,§ 13 des Münzgesetzes vom 16. Dezember 1999 (B...
9,1-DM-GoldmünzG,§ 10,Errichtung der Stiftung,"(1) Unter dem Namen ""Geld und Währung"" wird ei..."


In [10]:
def download_law(slug):
    law_dir = DATA / "laws" / slug
    if not law_dir.exists():
        link = f"https://www.gesetze-im-internet.de/{slug}/xml.zip"
        r = requests.get(link, headers=HEADERS, timeout=30)
        r.raise_for_status()  # bricht bei Fehlern wie 404 ab
        law_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall(law_dir)
    return next(law_dir.glob("*.xml"))

gmbh = parse_law(download_law("gmbhg"))
print(len(gmbh), "Paragrafen")
gmbh.head(10)

127 Paragrafen


,jurabk,enbez,titel,text
0,GmbHG,Inhaltsübersicht,NaN,Abschnitt 1 Errichtung der Gesellschaft § 1 Zw...
1,GmbHG,§ 1,Zweck; Gründerzahl,Gesellschaften mit beschränkter Haftung können...
2,GmbHG,§ 2,Form des Gesellschaftsvertrags,(1) Der Gesellschaftsvertrag bedarf notarielle...
3,GmbHG,§ 3,Inhalt des Gesellschaftsvertrags,(1) Der Gesellschaftsvertrag muß enthalten: 1....
4,GmbHG,§ 4,Firma,"Die Firma der Gesellschaft muß, auch wenn sie ..."
5,GmbHG,§ 4a,Sitz der Gesellschaft,"Sitz der Gesellschaft ist der Ort im Inland, d..."
6,GmbHG,§ 5,Stammkapital; Geschäftsanteil,(1) Das Stammkapital der Gesellschaft muß mind...
7,GmbHG,§ 5a,Unternehmergesellschaft,"(1) Eine Gesellschaft, die mit einem Stammkapi..."
8,GmbHG,§ 6,Geschäftsführer,(1) Die Gesellschaft muß einen oder mehrere Ge...
9,GmbHG,§ 7,Anmeldung der Gesellschaft,"(1) Die Gesellschaft ist bei dem Gericht, in d..."


In [11]:
gmbh["enbez"].str.split().str[0].value_counts()

enbez
§                   124
Anlage                2
Inhaltsübersicht      1
Name: count, dtype: int64

In [19]:
SKIP = {"Inhaltsübersicht"}

def parse_law(xml_path, slug=None):
    tree = etree.parse(str(xml_path))
    rows = []
    for norm in tree.getroot().findall("norm"):
        enbez = norm.findtext("metadaten/enbez")
        if enbez is None or enbez.strip() in SKIP:
            continue  # Kopf, Abschnittsüberschrift oder Inhaltsverzeichnis

        content = norm.find("textdaten/text/Content")
        if content is None:
            continue

        text = " ".join(" ".join(content.itertext()).split())
        if not text:
            continue  # leere Einträge überspringen

        rows.append({
            "slug": slug,
            "jurabk": norm.findtext("metadaten/jurabk"),
            "enbez": enbez.strip(),
            "typ": enbez.strip().split()[0],
            "titel": norm.findtext("metadaten/titel"),
            "text": text,
        })
    return pd.DataFrame(rows)

gmbh = parse_law(download_law("gmbhg"), "gmbhg")
print(len(gmbh), "Paragrafen")
gmbh["enbez"].str.split().str[0].value_counts()

116 Paragrafen


enbez
§         114
Anlage      2
Name: count, dtype: int64

In [20]:
alt = gmbh["text"].str.count(r"\bmuß\b").sum()
neu = gmbh["text"].str.count(r"\bmuss\b").sum()
print("muß:", alt, "| muss:", neu)

muß: 13 | muss: 12


In [21]:
def parse_all_norms(xml_path):
    rows = []
    for norm in etree.parse(str(xml_path)).getroot().findall("norm"):
        content = norm.find("textdaten/text/Content")
        text = "" if content is None else " ".join(" ".join(content.itertext()).split())
        rows.append({
            "enbez": norm.findtext("metadaten/enbez"),
            
            "titel": norm.findtext("metadaten/titel"),
            "text": text,
        })
    return pd.DataFrame(rows)

alle = parse_all_norms(download_law("gmbhg"))
alle[alle["enbez"].notna() & (alle["text"] == "")]

,enbez,titel,text
25,§ 17,(weggefallen),
41,§ 32a,(weggefallen),
42,§ 32b,(weggefallen),
78,§ 57b,(weggefallen),
99,§ 59,(weggefallen),
104,§ 63,(weggefallen),
105,§ 64,(weggefallen),
124,§ 80,(weggefallen),
125,§ 81,(weggefallen),
127,§ 83,(weggefallen),


In [22]:
gmbh["typ"].value_counts()

typ
§         114
Anlage      2
Name: count, dtype: int64

In [23]:
len(list((DATA / "laws").iterdir()))

274

In [24]:
from tqdm import tqdm

law_dirs = sorted(p for p in (DATA / "laws").iterdir() if p.is_dir())
print(len(law_dirs), "Gesetze bisher heruntergeladen")

frames, problems = [], []
for d in tqdm(law_dirs):
    xmls = list(d.glob("*.xml"))
    if not xmls:
        problems.append((d.name, "keine XML-Datei"))
        continue
    try:
        frames.append(parse_law(xmls[0], d.name))
    except Exception as e:
        problems.append((d.name, repr(e)))

corpus = pd.concat(frames, ignore_index=True)
ohne_paragrafen = sorted(set(d.name for d in law_dirs) - set(corpus["slug"]))

print("Gesetze mit Paragrafen:", corpus["slug"].nunique())
print("Gesetze ohne Paragrafen:", len(ohne_paragrafen))
print("Paragrafen gesamt:", len(corpus))
print("Probleme:", len(problems))

333 Gesetze bisher heruntergeladen


100%|███████████████████████████████████████| 333/333 [00:00<00:00, 679.24it/s]

Gesetze mit Paragrafen: 333
Gesetze ohne Paragrafen: 0
Paragrafen gesamt: 6233
Probleme: 0


In [25]:
corpus["typ"].value_counts().head(20)

typ
§                     5320
Anlage                 362
Eingangsformel         199
Art                    112
Anhang                 100
Schlussformel           66
Schlußformel            35
(XXXX)                  28
Artikel                  4
Inhaltsverzeichnis       2
I.                       1
II.                      1
III.                     1
Präambel                 1
Übersicht                1
Name: count, dtype: int64

In [26]:
corpus.loc[corpus["typ"] == "(XXXX)", ["slug", "enbez", "titel", "text"]].head(10)

,slug,enbez,titel,text
504,abgg10u11abschnabestbek,(XXXX),None,Der Ältestenrat des Deutschen Bundestages hat ...
2342,albeitrbek_2012,(XXXX),None,"Auf Grund des § 33 Absatz 1 und der §§ 68, 114..."
2344,albeitrbek_2013,(XXXX),None,"Auf Grund des § 33 Absatz 1 und der §§ 68, 114..."
2346,albeitrbek_2014,(XXXX),None,"Auf Grund des § 33 Absatz 1 und der §§ 68, 114..."
2348,albeitrbek_2015,(XXXX),None,"Auf Grund des § 33 Absatz 1 und der §§ 68, 114..."
2350,albeitrbek_2016,(XXXX),None,"Auf Grund des § 33 Absatz 1 und der §§ 68, 114..."
2352,albeitrbek_2017,(XXXX),None,"Auf Grund des § 33 Absatz 1 und der §§ 68, 114..."
2354,albeitrbek_2018,(XXXX),None,"Auf Grund des § 33 Absatz 1 und der §§ 68, 114..."
2356,albeitrbek_2019,(XXXX),None,"Auf Grund des § 33 Absatz 1 und der §§ 68, 114..."
2358,albeitrbek_2020,(XXXX),None,"Auf Grund des § 33 Absatz 1 und der §§ 68, 114..."


In [29]:
SKIP = {
    "Inhaltsübersicht", "Inhaltsverzeichnis", "Übersicht",
    "Eingangsformel", "Schlussformel", "Schlußformel", "Präambel",
}

def parse_law(xml_path, slug=None):
    tree = etree.parse(str(xml_path))
    rows = []
    for norm in tree.getroot().findall("norm"):
        enbez = norm.findtext("metadaten/enbez")
        if enbez is None:
            continue  # Kopf oder Abschnittsüberschrift
        enbez = enbez.strip()
        typ = enbez.split()[0]
        if typ in SKIP:
            continue  # Formeln und Inhaltsverzeichnisse

        content = norm.find("textdaten/text/Content")
        if content is None:
            continue

        text = " ".join(" ".join(content.itertext()).split())
        if not text:
            continue

        rows.append({
            "slug": slug,
            "jurabk": norm.findtext("metadaten/jurabk"),
            "enbez": enbez,
            "typ": typ,
            "titel": norm.findtext("metadaten/titel"),
            "text": text,
        })
    return pd.DataFrame(rows)

In [30]:
print(SKIP)

{'Schlußformel', 'Inhaltsübersicht', 'Eingangsformel', 'Präambel', 'Übersicht', 'Inhaltsverzeichnis', 'Schlussformel'}


In [31]:
corpus["typ"].value_counts().head(20)

typ
§                     5320
Anlage                 362
Eingangsformel         199
Art                    112
Anhang                 100
Schlussformel           66
Schlußformel            35
(XXXX)                  28
Artikel                  4
Inhaltsverzeichnis       2
I.                       1
II.                      1
III.                     1
Präambel                 1
Übersicht                1
Name: count, dtype: int64

In [32]:
law_dirs = sorted(p for p in (DATA / "laws").iterdir() if p.is_dir())
print(len(law_dirs), "Gesetze bisher heruntergeladen")

frames, problems = [], []
for d in tqdm(law_dirs):
    xmls = list(d.glob("*.xml"))
    if not xmls:
        problems.append((d.name, "keine XML-Datei"))
        continue
    try:
        frames.append(parse_law(xmls[0], d.name))
    except Exception as e:
        problems.append((d.name, repr(e)))

corpus = pd.concat(frames, ignore_index=True)
print("Paragrafen gesamt:", len(corpus))
print("Probleme:", len(problems))

corpus["typ"].value_counts().head(20)

576 Gesetze bisher heruntergeladen


100%|███████████████████████████████████████| 576/576 [00:00<00:00, 707.49it/s]

Paragrafen gesamt: 10235
Probleme: 0


typ
§             9292
Anlage         558
Art            192
Anhang         108
(XXXX)          67
Artikel          4
I.               2
II.              2
Gliederung       2
III.             1
1.               1
2.               1
3.               1
4.               1
5.               1
6.               1
Muster           1
Name: count, dtype: int64

In [33]:
CORE = {"§", "Art", "Artikel"}
ANNEX = {"Anlage", "Anhang", "Muster"}

def gruppe(typ):
    if typ in CORE:
        return "Vorschrift"
    if typ in ANNEX:
        return "Anhang"
    if typ == "(XXXX)":
        return "ohne Nummer"
    return "sonstige"

corpus["gruppe"] = corpus["typ"].map(gruppe)
corpus["gruppe"].value_counts()

gruppe
Vorschrift     9488
Anhang          667
ohne Nummer      67
sonstige         13
Name: count, dtype: int64

In [34]:
corpus.to_csv(DATA / "corpus.csv", index=False)

In [35]:
toc = pd.read_csv(DATA / "toc.csv")
law_dirs = sorted(p for p in (DATA / "laws").iterdir() if p.is_dir())
print(len(law_dirs), "von", len(toc), "Gesetzen heruntergeladen")

frames, problems = [], []
for d in tqdm(law_dirs):
    xmls = list(d.glob("*.xml"))
    if not xmls:
        problems.append((d.name, "keine XML-Datei"))
        continue
    try:
        frames.append(parse_law(xmls[0], d.name))
    except Exception as e:
        problems.append((d.name, repr(e)))

corpus = pd.concat(frames, ignore_index=True)
corpus["gruppe"] = corpus["typ"].map(gruppe)

print("Gesetze mit Paragrafen:", corpus["slug"].nunique())
print("Paragrafen gesamt:", len(corpus))
print("Probleme:", len(problems))
print()
print(corpus["gruppe"].value_counts())

6144 von 6144 Gesetzen heruntergeladen


100%|█████████████████████████████████████| 6144/6144 [00:07<00:00, 798.22it/s]


Gesetze mit Paragrafen: 5965
Paragrafen gesamt: 100776
Probleme: 0

gruppe
Vorschrift     93718
Anhang          4891
ohne Nummer     1423
sonstige         744
Name: count, dtype: int64


In [36]:
corpus.loc[corpus["typ"] == "Gliederung", ["slug", "enbez", "titel", "text"]]

,slug,enbez,titel,text
6596,auslschuldabkag,Gliederung,None,Erster Abschnitt Begriffsbestimmungen ...........
6759,auslwbg,Gliederung,NaN,Abschnitt I Allgemeine Grundzüge des Bereinigu...


In [37]:
corpus.to_csv(DATA / "corpus.csv", index=False)

In [38]:
SKIP.add("Gliederung")

In [39]:
ohne = toc[~toc["slug"].isin(corpus["slug"])]
print(len(ohne), "Gesetze ohne Paragrafen")
ohne["title"].head(15).tolist()

179 Gesetze ohne Paragrafen


['Gesetz zur Neuordnung des Arzneimittelrechts',
 'Rechtsverordnung des Präsidenten des Bundesausgleichsamtes zur Einführung von Rechtsverordnungen im Saarland',
 'Gesetz zur Neuregelung der Vorschriften über den Bundesgrenzschutz',
 'Gesetz über den Eintritt der Freistaaten Bayern und Baden in die Biersteuergemeinschaft',
 'Gesetz über den Eintritt des Freistaats Württemberg in die Biersteuergemeinschaft',
 'Verordnung über die Entgelte für die Leistungen der Binnenlotsen auf der Bundeswasserstraße Rhein zwischen den Schleusen Iffezheim und Mannheim/Ludwigshafen',
 'Bekanntmachung des Organisationserlasses des Bundeskanzlers',
 'Bekanntmachung des Organisationserlasses des Bundeskanzlers',
 'Bau- und Betriebsordnung für Pioniereisenbahnen',
 'Verwaltungsanordnung über die Einrichtung des Bundesoberseeamts',
 'Abkommen zwischen dem Deutschen Institut in Taipeh und der Taipeh Vertretung in der Bundesrepublik Deutschland zur Vermeidung der Doppelbesteuerung und zur Verhinderung der Steue

In [40]:
corpus.to_csv(DATA / "corpus.csv", index=False)

In [41]:
def text_ohne_enbez(xml_path):
    total = 0
    for norm in etree.parse(str(xml_path)).getroot().findall("norm"):
        if norm.findtext("metadaten/enbez") is None:
            content = norm.find("textdaten/text/Content")
            if content is not None:
                total += len(" ".join(" ".join(content.itertext()).split()))
    return total

ohne = ohne.copy()
ohne["zeichen"] = [
    text_ohne_enbez(next((DATA / "laws" / s).glob("*.xml"))) for s in ohne["slug"]
]

print((ohne["zeichen"] == 0).sum(), "Gesetze ganz ohne Text")
print((ohne["zeichen"] > 0).sum(), "Gesetze mit Text ohne Nummer")
ohne.sort_values("zeichen", ascending=False)[["slug", "title", "zeichen"]].head(10)

168 Gesetze ganz ohne Text
11 Gesetze mit Text ohne Nummer


,slug,title,zeichen
4232,pflegevg,Gesetz zur sozialen Absicherung des Risikos de...,15175
1964,europolg,Gesetz zur Anwendung der Verordnung (EU) 2016/...,8190
2498,gr_ndstvtrbb_st2024bek,Bekanntmachung über den Abschluss und das Inkr...,1647
2956,kfbg,Gesetz zur Bereinigung von Kriegsfolgengesetzen,1466
1856,erbstrg,Gesetz zur Reform des Erbschaftsteuer- und Sch...,1436
338,amrnog,Gesetz zur Neuordnung des Arzneimittelrechts,1323
4550,rv_uvrag_1987,Gesetz über die Anpassung der Renten der geset...,1279
2652,hochseeschginkrbek,Bekanntmachung über das Inkrafttreten des Hoch...,561
5083,stvgart9s1inkrbek,Bekanntmachung über das Inkrafttreten der Arti...,518
2258,fzulbvinkrbek,Bekanntmachung über das Inkrafttreten der Fors...,272


In [42]:
fehlend = ohne["zeichen"].sum()
gesamt = corpus["text"].str.len().sum()

print((ohne["zeichen"] == 0).sum(), "Gesetze ganz ohne Text")
print(f"{fehlend:,} fehlende Zeichen von {gesamt:,} im Korpus")
print(f"Anteil: {fehlend / gesamt:.3%}")

168 Gesetze ganz ohne Text
32,106 fehlende Zeichen von 151,872,401 im Korpus
Anteil: 0.021%
